<a href="https://colab.research.google.com/github/hibahrehman25-lang/ML_inter_Task1/blob/main/Copy_of_w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hibahrehman25-lang/ML_inter_Task1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### setup 0

In [23]:
import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

HF_TOKEN = userdata.get("HF_TOKEN")

print("Imports successful")
print("HF token loaded:", HF_TOKEN is not None)

Imports successful
HF token loaded: True


In [24]:
import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata

# Load Hugging Face token
HF_TOKEN = userdata.get("HF_TOKEN")

print("HF token loaded:", HF_TOKEN is not None)

# Connect to DuckDB
con = duckdb.connect()

# Authenticate with Hugging Face
con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("DuckDB connected and Hugging Face authentication configured.")

HF token loaded: True
DuckDB connected and Hugging Face authentication configured.


In [25]:
model_df = con.sql("""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS total_impressions,
    SUM(gsc_clicks) AS total_clicks,
    AVG(gsc_avg_position) AS avg_position,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN (SUM(gsc_clicks) * 100.0) / SUM(gsc_impressions)
        ELSE NULL
    END AS ctr

FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'

WHERE
    gsc_data_available = TRUE
    AND gsc_impressions > 0
    AND gsc_avg_position > 0

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

print("Model rows:", len(model_df))
print("Unique clients:", model_df["client_hash_id"].nunique())

print("Model rows:", len(model_df))
print("Unique clients:", model_df["client_hash_id"].nunique())
print("\nColumns:")
print(model_df.columns.tolist())

print("\nFirst 5 rows:")
display(model_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Model rows: 175304
Unique clients: 47
Model rows: 175304
Unique clients: 47

Columns:
['client_hash_id', 'content_hash_id', 'total_impressions', 'total_clicks', 'avg_position', 'ctr']

First 5 rows:


,client_hash_id,content_hash_id,total_impressions,total_clicks,avg_position,ctr
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,0.107313
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,424.0,0.0,3.307255,0.000000
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,0.106572
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,0.262945
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,417.0,1.0,4.499519,0.239808


In [26]:
# Week-5 opportunity label
impression_threshold = model_df["total_impressions"].quantile(0.75)

model_df["opportunity_label"] = (
    (model_df["total_impressions"] >= impression_threshold)
    & (model_df["total_clicks"] == 0)
).astype(int)

print("75th percentile impression threshold:", impression_threshold)

print("\nOpportunity label counts:")
print(model_df["opportunity_label"].value_counts())

print("\nOpportunity label proportions:")
print(model_df["opportunity_label"].value_counts(normalize=True))


75th percentile impression threshold: 1052.0

Opportunity label counts:
opportunity_label
0    171247
1      4057
Name: count, dtype: int64

Opportunity label proportions:
opportunity_label
0    0.976857
1    0.023143
Name: proportion, dtype: float64


In [27]:
# Step 5: Week-5 features

# 1. Log-transformed impressions
model_df["log_impressions"] = np.log1p(
    model_df["total_impressions"]
)

# 2. Position inverse
model_df["position_inverse"] = (
    1 / (model_df["avg_position"] + 1)
)

# 3. Position buckets
position_bins = [0, 3, 10, 20, np.inf]
position_labels = ["1-3", "4-10", "11-20", "21+"]

model_df["position_bucket"] = pd.cut(
    model_df["avg_position"],
    bins=position_bins,
    labels=position_labels,
    include_lowest=True
)

# 4. Median CTR for each position bucket
bucket_ctr = (
    model_df
    .groupby("position_bucket", observed=False)["ctr"]
    .median()
    .reset_index(name="bucket_median_ctr")
)

# 5. Merge bucket median CTR
model_df = model_df.merge(
    bucket_ctr,
    on="position_bucket",
    how="left"
)

# 6. CTR gap from bucket median
model_df["ctr_gap"] = (
    model_df["bucket_median_ctr"] - model_df["ctr"]
)

features = [
    "log_impressions",
    "avg_position",
    "position_inverse",
    "ctr_gap"
]

print("Week-5 features:")
print(features)

print("\nFeature preview:")
display(model_df[features + ["opportunity_label"]].head())

print("\nMissing values:")
print(model_df[features].isna().sum())

Week-5 features:
['log_impressions', 'avg_position', 'position_inverse', 'ctr_gap']

Feature preview:


,log_impressions,avg_position,position_inverse,ctr_gap,opportunity_label
0,8.783243,7.209549,0.121809,-0.107313,0
1,6.052089,3.307255,0.232166,0.000000,0
2,8.636042,6.724039,0.129466,-0.106572,0
3,8.506132,7.244844,0.121288,-0.262945,0
4,6.035481,4.499519,0.181834,-0.239808,0



Missing values:
log_impressions     0
avg_position        0
position_inverse    0
ctr_gap             0
dtype: int64


In [28]:
# Step 6: Leakage Audit
# Check how each Week-5 feature is related to the opportunity label.

leakage_check = pd.DataFrame({
    "feature": [
        "log_impressions",
        "avg_position",
        "position_inverse",
        "ctr_gap"
    ],
    "source_column": [
        "total_impressions",
        "avg_position",
        "avg_position",
        "ctr + position_bucket"
    ],
    "used_in_label_definition": [
        True,
        False,
        False,
        True
    ],
    "derived_from_label_inputs": [
        True,
        False,
        False,
        True
    ],
    "audit_status": [
        "LEAKAGE RISK",
        "TIMING CHECK",
        "TIMING CHECK",
        "LEAKAGE RISK"
    ]
})

display(leakage_check)

,feature,source_column,used_in_label_definition,derived_from_label_inputs,audit_status
0,log_impressions,total_impressions,True,True,LEAKAGE RISK
1,avg_position,avg_position,False,False,TIMING CHECK
2,position_inverse,avg_position,False,False,TIMING CHECK
3,ctr_gap,ctr + position_bucket,True,True,LEAKAGE RISK


In [29]:
# Quantify how strongly each feature separates the two label classes.

feature_signal = (
    model_df
    .groupby("opportunity_label")[features]
    .mean()
    .T
)

feature_signal.columns = ["label_0_mean", "label_1_mean"]

feature_signal["absolute_difference"] = (
    feature_signal["label_1_mean"]
    - feature_signal["label_0_mean"]
).abs()

display(feature_signal.sort_values(
    "absolute_difference",
    ascending=False
))

,label_0_mean,label_1_mean,absolute_difference
log_impressions,4.939164,7.535687,2.596523
avg_position,17.031164,17.869047,0.837883
ctr_gap,-0.421417,0.005758,0.427175
position_inverse,0.113852,0.100568,0.013284


In [30]:
# Step 7: Week-5 random split (BEFORE)

from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

X = model_df[features]
y = model_df["opportunity_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nTraining positive rate:", y_train.mean())
print("Testing positive rate:", y_test.mean())

print("\nWeek-5 random split:")
print("Train clients:", model_df.loc[X_train.index, "client_hash_id"].nunique())
print("Test clients:", model_df.loc[X_test.index, "client_hash_id"].nunique())

client_overlap = set(
    model_df.loc[X_train.index, "client_hash_id"]
).intersection(
    set(model_df.loc[X_test.index, "client_hash_id"])
)

print("Clients appearing in BOTH train and test:", len(client_overlap))

Training rows: 140243
Testing rows: 35061

Training positive rate: 0.023145540240867638
Testing positive rate: 0.02313111434357263

Week-5 random split:
Train clients: 46
Test clients: 46
Clients appearing in BOTH train and test: 45


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two paper findings + my methodology questions

### Finding 1 — Click Capture by Position Tier

The paper reports that weighted CTR decreases as visibility moves away from the top of search results. The measured outcome is weighted CTR, calculated from clicks and impressions within each position tier.

**Methodology question:** Does this observational comparison support the stronger interpretation that improving page-one visibility will produce incremental clicks, or would a time-based or matched comparison be needed to support that claim more strongly?

### Finding 2 — The Freshness Multiplier

The paper reports higher measured health and impressions for mature pages that were refreshed compared with non-refreshed pages. The paper also notes that the 361+ freshness bucket is small and unstable.

**Methodology question:** Could differences in prior visibility, age, topic, or page quality partly explain the observed gap between refreshed and non-refreshed pages? A matched or pre/post comparison could provide stronger evidence about the relationship.

In [31]:
# Section 1: supporting check for the two paper findings

paper_findings_check = pd.DataFrame({
    "Finding": [
        "Click Capture by Position Tier",
        "Freshness Multiplier"
    ],
    "Measured_outcome": [
        "Weighted CTR by position tier",
        "Measured health and impressions by freshness/refresh status"
    ],
    "Methodology_question": [
        "Does the observational design support an incremental-click interpretation?",
        "Could selection differences explain part of the observed gap?"
    ]
})

display(paper_findings_check)


,Finding,Measured_outcome,Methodology_question
0,Click Capture by Position Tier,Weighted CTR by position tier,Does the observational design support an incre...
1,Freshness Multiplier,Measured health and impressions by freshness/r...,Could selection differences explain part of th...


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### 2. My model under an honest split (before/after)

In Week 5, I used a random stratified 80/20 split. The positive-class rate was preserved, but 46 clients appeared in both the training and test sets. This means the evaluation was not independent at the client level.

For the honest validation check, I used a grouped-by-client split. The grouped split produced zero client overlap between training and test sets.

The measured F1 score was 0.9505 on the Week-5 random split and 0.9514 on the grouped split. Precision decreased from 0.9656 to 0.9544, while recall increased from 0.9359 to 0.9485.

The grouped result did not show a large performance drop. It provides a more client-independent validation estimate, but it does not by itself establish future predictive performance.

In [32]:
# Section 2: Before vs After validation

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

# Week-5 features and label
X = model_df[features]
y = model_df["opportunity_label"]

# -----------------------------
# BEFORE: Week-5 random split
# -----------------------------

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

random_model = HistGradientBoostingClassifier(
    random_state=42,
    max_depth=6,
    learning_rate=0.1,
    max_iter=200
)

random_model.fit(X_train_random, y_train_random)
random_pred = random_model.predict(X_test_random)

random_results = {
    "Split": "Week-5 Random Split",
    "Precision": precision_score(y_test_random, random_pred, zero_division=0),
    "Recall": recall_score(y_test_random, random_pred, zero_division=0),
    "F1": f1_score(y_test_random, random_pred, zero_division=0),
    "Accuracy": accuracy_score(y_test_random, random_pred),
    "Positive base rate": y_test_random.mean()
}

# -----------------------------
# AFTER: Grouped-by-client split
# -----------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_df,
        y,
        groups=model_df["client_hash_id"]
    )
)

group_train = model_df.iloc[train_idx]
group_test = model_df.iloc[test_idx]

X_train_group = group_train[features]
X_test_group = group_test[features]

y_train_group = group_train["opportunity_label"]
y_test_group = group_test["opportunity_label"]

group_model = HistGradientBoostingClassifier(
    random_state=42,
    max_depth=6,
    learning_rate=0.1,
    max_iter=200
)

group_model.fit(X_train_group, y_train_group)
group_pred = group_model.predict(X_test_group)

group_results = {
    "Split": "Grouped by Client",
    "Precision": precision_score(y_test_group, group_pred, zero_division=0),
    "Recall": recall_score(y_test_group, group_pred, zero_division=0),
    "F1": f1_score(y_test_group, group_pred, zero_division=0),
    "Accuracy": accuracy_score(y_test_group, group_pred),
    "Positive base rate": y_test_group.mean()
}

# -----------------------------
# Comparison
# -----------------------------

validation_comparison = pd.DataFrame([
    random_results,
    group_results
])

display(validation_comparison)

print(
    "Random split client overlap:",
    len(
        set(model_df.loc[X_train_random.index, "client_hash_id"])
        & set(model_df.loc[X_test_random.index, "client_hash_id"])
    )
)

print(
    "Grouped split client overlap:",
    len(
        set(group_train["client_hash_id"])
        & set(group_test["client_hash_id"])
    )
)

,Split,Precision,Recall,F1,Accuracy,Positive base rate
0,Week-5 Random Split,0.983770,0.971640,0.977667,0.998973,0.023131
1,Grouped by Client,0.942446,0.963235,0.952727,0.997959,0.021356


Random split client overlap: 45
Grouped split client overlap: 0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage audit

I audited the final Week-5 feature set against the way the opportunity label was constructed.

`log_impressions` is a leakage risk because it is directly derived from `total_impressions`, and total impressions are part of the label definition.

`ctr_gap` is also a leakage risk because it uses CTR, which is calculated from clicks and impressions, while clicks are directly used in the label definition.

`avg_position` and `position_inverse` are not direct label inputs, but they are measured from the same March observation window as the label. They therefore require a timing check if the intended task is future prediction.

The audit suggests that the Week-5 model should not be interpreted as a clean future-outcome predictor. Its measured performance is better described as performance on reproducing the current opportunity label.

In [33]:
# Section 3: Leakage audit on the final Week-5 feature set

leakage_audit = pd.DataFrame({
    "Feature": [
        "log_impressions",
        "avg_position",
        "position_inverse",
        "ctr_gap"
    ],
    "Source": [
        "total_impressions",
        "avg_position",
        "avg_position",
        "ctr + position_bucket"
    ],
    "Label_dependency": [
        "Direct: impressions define part of the label",
        "No direct dependency",
        "No direct dependency",
        "Direct/sibling: CTR uses clicks and impressions"
    ],
    "Audit_status": [
        "LEAKAGE RISK",
        "TIMING CHECK",
        "TIMING CHECK",
        "LEAKAGE RISK"
    ]
})

display(leakage_audit)

print("\nLabel definition:")
print(
    "opportunity_label = "
    "(total_impressions >= 75th percentile) AND "
    "(total_clicks == 0)"
)

print("\nPositive base rate:",
      round(model_df["opportunity_label"].mean(), 6))


,Feature,Source,Label_dependency,Audit_status
0,log_impressions,total_impressions,Direct: impressions define part of the label,LEAKAGE RISK
1,avg_position,avg_position,No direct dependency,TIMING CHECK
2,position_inverse,avg_position,No direct dependency,TIMING CHECK
3,ctr_gap,ctr + position_bucket,Direct/sibling: CTR uses clicks and impressions,LEAKAGE RISK



Label definition:
opportunity_label = (total_impressions >= 75th percentile) AND (total_clicks == 0)

Positive base rate: 0.023143


### Error examples

The grouped-by-client test set was used to inspect individual classification failures.

A false positive is a page that the model classified as an opportunity but whose observed label was 0. A false negative is a page whose observed label was 1 but the model classified it as 0.

These examples are used for interpretation and debugging, not as evidence of causality.

In [34]:
# Inspect real failure examples from the grouped-by-client test set

error_examples = group_test[
    ["client_hash_id", "content_hash_id",
     "total_impressions", "total_clicks",
     "avg_position", "ctr", "opportunity_label"]
].copy()

error_examples["prediction"] = group_pred

# False positives
false_positives = error_examples[
    (error_examples["opportunity_label"] == 0) &
    (error_examples["prediction"] == 1)
].copy()

# False negatives
false_negatives = error_examples[
    (error_examples["opportunity_label"] == 1) &
    (error_examples["prediction"] == 0)
].copy()

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nFalse-positive examples:")
display(
    false_positives[
        ["total_impressions", "total_clicks",
         "avg_position", "ctr",
         "opportunity_label", "prediction"]
    ].head(5)
)

print("\nFalse-negative examples:")
display(
    false_negatives[
        ["total_impressions", "total_clicks",
         "avg_position", "ctr",
         "opportunity_label", "prediction"]
    ].head(5)
)

False positives: 48
False negatives: 30

False-positive examples:


,total_impressions,total_clicks,avg_position,ctr,opportunity_label,prediction
819,1332.0,1.0,2.341058,0.075075,0,1
1920,1439.0,1.0,2.679433,0.069493,0,1
2682,1527.0,1.0,2.100409,0.065488,0,1
3285,1257.0,1.0,0.895386,0.079554,0,1
3331,1829.0,1.0,2.425843,0.054675,0,1



False-negative examples:


,total_impressions,total_clicks,avg_position,ctr,opportunity_label,prediction
1116,4225.0,0.0,1.913571,0.0,1,0
1876,2323.0,0.0,2.137640,0.0,1,0
2137,28950.0,0.0,9.419929,0.0,1,0
2709,2409.0,0.0,1.532699,0.0,1,0
3198,9887.0,0.0,1.953104,0.0,1,0


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### 4. Claim rewrite

### Original Week-5 claim

The model can identify content pages that are likely to be high-priority CTR opportunities.

### Safer rewritten claim

On this March 2026 dataset, the model measured high classification performance for reproducing the defined opportunity label under both a random split and a client-grouped split. The grouped evaluation had zero client overlap with training data and produced an F1 score of 0.9514.

This result is directional decision-support evidence for prioritizing pages that match the defined opportunity rule. It does not establish that the model will predict future CTR improvement or that acting on these pages will cause additional clicks.

In [35]:
# Section 4: Supporting numbers for the rewritten claim

claim_check = pd.DataFrame({
    "Measure": [
        "Dataset rows",
        "Unique clients",
        "Positive opportunities",
        "Positive rate",
        "Random split F1",
        "Grouped-by-client F1",
        "Grouped client overlap"
    ],
    "Observed_value": [
        len(model_df),
        model_df["client_hash_id"].nunique(),
        int(model_df["opportunity_label"].sum()),
        round(model_df["opportunity_label"].mean(), 6),
        round(random_results["F1"], 4),
        round(group_results["F1"], 4),
        len(
            set(group_train["client_hash_id"])
            & set(group_test["client_hash_id"])
        )
    ]
})

display(claim_check)


,Measure,Observed_value
0,Dataset rows,175304.000000
1,Unique clients,47.000000
2,Positive opportunities,4057.000000
3,Positive rate,0.023143
4,Random split F1,0.977700
5,Grouped-by-client F1,0.952700
6,Grouped client overlap,0.000000


## Self-check

Before you submit, confirm each line honestly:

- [✔️] Every section above is filled — markdown thinking AND the code that backs it
- [✔️] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔️] No client names, URLs, or private queries anywhere
- [✔️] My claims use careful words: observed, measured, directional, decision-support
- [✔️] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.